In [1]:
import sys
import os
import torch.nn.functional as F
project_root = os.path.abspath(os.path.join(os.getcwd(),'..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)
from src.processor.word_process import sudachi_tokenize
import pandas as pd
import numpy as np
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm
import torch



/home/manaty/company-classification-analysis/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_csv("../data/raw/edinet_analysis_data_listed_only.csv")
print(df.shape)
df.head()

(3637, 9)


,company_name,証券コード,33業種コード,33業種区分,17業種コード,17業種区分,business_description,business_risks,filename
0,AGC株式会社,5201,3400,ガラス・土石製品,3,建設・資材,3 事業の内容 当社及び当社の子会社(以下、「当社グループ」という。)並びに当社の関連会社は...,3 事業等のリスク (1)リスクマネジメント体制 当社グループでは、「第4 提出会社の状況 ...,S100XSTU.zip
1,AGS株式会社,3648,5250,情報・通信業,10,情報通信・サービスその他,3 事業の内容 当社グループ(当社及び当社の関係会社)は、当社と連結子会社3社とで構成されて...,3 事業等のリスク 文中の将来に関する事項は、当連結会計年度末現在において当社グループが判断...,S100W007.zip
2,AHCグループ株式会社,7083,9050,サービス業,10,情報通信・サービスその他,3 事業の内容 当社グループは、当社、連結子会社(SLカンパニー株式会社、テラスワールド株式...,3 事業等のリスク 有価証券報告書に記載した事業の状況、経理の状況等に関する事項のうち、投資...,S100XMZ1.zip
3,AI CROSS株式会社,4476,5250,情報・通信業,10,情報通信・サービスその他,"3 事業の内容 当社グループは、「Smart Work, Smart Life~人生のいい時...",3 事業等のリスク 本書に記載した事業の状況、経理の状況等に関する事項のうち、投資家の判断に...,S100XUJV.zip
4,AI inside 株式会社,4488,5250,情報・通信業,10,情報通信・サービスその他,3 事業の内容 当社は、「AIで、人類の進化と人々の幸福に貢献する」というパーパスのもと、「...,3 事業等のリスク 当社は、事業展開上のリスクになる可能性があると考えられる主な要因として、...,S100W5FA.zip


In [3]:
#チャンク化
from src.processor.text_processor import chunk_text_by_sentence
chunked_data = []
for idx, row in df.iterrows():
    chunks = chunk_text_by_sentence(row["business_description"])
    for chunk in chunks:
        chunked_data.append({
            "company_name": row["company_name"],
            "chunk_text": f"query: {chunk}"
        })

df_chunks = pd.DataFrame(chunked_data)
print(f"全チャンク数: {len(df_chunks)}")
df_chunks.head()

全チャンク数: 12442


,company_name,chunk_text
0,AGC株式会社,query: 3 事業の内容 当社及び当社の子会社(以下、「当社グループ」という。)並びに当...
1,AGS株式会社,query: 3 事業の内容 当社グループ(当社及び当社の関係会社)は、当社と連結子会社3社...
2,AGS株式会社,query: (2) ソフトウエア開発 長年にわたるソリューション提供の実績とエンジニアリン...
3,AGS株式会社,query: (*1) クラウドサービスとは、データセンターのハードウエア資源やアプリケーシ...
4,AHCグループ株式会社,query: 3 事業の内容 当社グループは、当社、連結子会社(SLカンパニー株式会社、テラ...


★SBERT（Sentence-BERT）の活用

In [4]:
"""
#ファインチューニング前のモデルによるベクトル化
#日本語対応のモデルをロード
model = SentenceTransformer('intfloat/multilingual-e5-base')
#企業の事業内容を一気にベクトル化(学習時間およそ60分)
sentences = df_chunks["chunk_text"].tolist()
business_embeddings = model.encode(sentences, show_progress_bar=True)
"""

'\n#ファインチューニング前のモデルによるベクトル化\n#日本語対応のモデルをロード\nmodel = SentenceTransformer(\'intfloat/multilingual-e5-base\')\n#企業の事業内容を一気にベクトル化(学習時間およそ60分)\nsentences = df_chunks["chunk_text"].tolist()\nbusiness_embeddings = model.encode(sentences, show_progress_bar=True)\n'

In [5]:
"""
df_chunks["embedding"] = list(business_embeddings)
grouped_series = df_chunks.groupby("company_name")["embedding"].apply(lambda x: np.mean(x.tolist(), axis=0))#ベクトルの平均化

df_final_embeddings = pd.DataFrame(
    grouped_series.tolist(),
    index=grouped_series.index)

final_embeddings = df_final_embeddings.values

print("企業名付きデータフレームの形状:", df_final_embeddings.shape) # (3637, 768)
print("NumPy配列の形状:", final_embeddings.shape)
df_final_embeddings.head()
"""

'\ndf_chunks["embedding"] = list(business_embeddings)\ngrouped_series = df_chunks.groupby("company_name")["embedding"].apply(lambda x: np.mean(x.tolist(), axis=0))#ベクトルの平均化\n\ndf_final_embeddings = pd.DataFrame(\n    grouped_series.tolist(),\n    index=grouped_series.index)\n\nfinal_embeddings = df_final_embeddings.values\n\nprint("企業名付きデータフレームの形状:", df_final_embeddings.shape) # (3637, 768)\nprint("NumPy配列の形状:", final_embeddings.shape)\ndf_final_embeddings.head()\n'

In [6]:
"""
#次回のために保存
np.save("../models/neo_company_embeddings.npy", final_embeddings)

# 1. 企業名付きデータフレーム（3637, 768）を pickle で保存する
df_final_embeddings.to_pickle('../models/neo_company_data_with_sbert.pkl')
"""

'\n#次回のために保存\nnp.save("../models/neo_business_embeddings.npy", final_embeddings)\n\n# 1. 企業名付きデータフレーム（3637, 768）を pickle で保存する\ndf_final_embeddings.to_pickle(\'../models/neo_company_data_with_sbert.pkl\')\n'

In [7]:
# 1. 育てたモデルのパスを指定（Classification用ではなく、素のAutoModelで読み込む）
model_path = "../models/e5_edinet_finetuned"
print("育てたモデルを読み込み中...")
tokenizer = AutoTokenizer.from_pretrained(model_path)
base_model = AutoModel.from_pretrained(model_path)

# GPUがあれば転送して爆速化（なければCPU）
device = "cuda" if torch.cuda.is_available() else "cpu"
base_model = base_model.to(device)
base_model.eval()

# 2. チャンクテキストのリストを取得
sentences = df_chunks["chunk_text"].tolist()

# 3. 平均プーリング（Mean Pooling）関数の定義
# E5/BERTの生出力を正しい768次元の文章ベクトルに変換するために必要です
def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output[0]
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)

# 4. バッチ処理で一気にベクトル化
batch_size = 32  # メモリに応じて調整してください
all_embeddings = []

print("ファインチューニング済みモデルでベクトル化を実行中...")
for i in tqdm(range(0, len(sentences), batch_size)):
    batch_sentences = sentences[i:i + batch_size]
    
    # トークナイズ
    encoded_input = tokenizer(
        batch_sentences, 
        padding=True, 
        truncation=True, 
        max_length=512, 
        return_tensors='pt'
    ).to(device)
    
    # ベクトル計算
    with torch.no_grad():
        model_output = base_model(**encoded_input)
        # トークンごとのベクトルから、文章全体の平均ベクトル（768次元）を抽出
        batch_embeddings = mean_pooling(model_output, encoded_input['attention_mask'])
        # CPUに書き戻してリストに追加
        all_embeddings.append(batch_embeddings.cpu().numpy())

# 特徴ベクトルを1つのNumPy配列に結合
business_embeddings = np.vstack(all_embeddings)
print("ベクトル化完了！ 形状:", business_embeddings.shape)

育てたモデルを読み込み中...


Loading weights: 100%|███████████████████████| 197/197 [00:00<00:00, 905.39it/s]
[transformers] XLMRobertaModel LOAD REPORT from: ../models/e5_edinet_finetuned
Key                        | Status     | 
---------------------------+------------+-
classifier.out_proj.bias   | UNEXPECTED | 
classifier.dense.bias      | UNEXPECTED | 
classifier.out_proj.weight | UNEXPECTED | 
classifier.dense.weight    | UNEXPECTED | 
pooler.dense.weight        | MISSING    | 
pooler.dense.bias          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


ファインチューニング済みモデルでベクトル化を実行中...


  3%|█▎                                      | 13/389 [02:05<1:00:39,  9.68s/it]


KeyboardInterrupt: 

In [28]:
df_chunks["embedding"] = list(business_embeddings)
grouped_series = df_chunks.groupby("company_name")["embedding"].apply(lambda x: np.mean(x.tolist(), axis=0))

df_final_embeddings = pd.DataFrame(
    grouped_series.tolist(),
    index=grouped_series.index)

final_embeddings = df_final_embeddings.values

print("企業名付きデータフレームの形状:", df_final_embeddings.shape) # (3637, 768)
print("NumPy配列の形状:", final_embeddings.shape)
df_final_embeddings.head()

企業名付きデータフレームの形状: (3637, 768)
NumPy配列の形状: (3637, 768)


,0,1,2,3,4,5,6,7,8,9,...,758,759,760,761,762,763,764,765,766,767
company_name,,,,,,,,,,,,,,,,,,,,,
AGC株式会社,-0.103023,0.094929,-0.235786,-0.234734,-0.228411,-0.056597,-0.437262,-0.109424,0.281605,0.137402,...,-0.512112,-0.275002,0.020890,-0.625053,-0.423893,-0.363505,0.117755,0.276579,-0.164813,-0.253370
AGS株式会社,-0.111819,0.039796,0.017986,-0.238876,-0.058648,-0.122806,0.242090,-0.115731,0.172139,-0.565255,...,-0.450986,0.378106,0.223879,0.036677,0.082660,-0.522888,0.217303,0.211664,-0.327770,-0.099943
AHCグループ株式会社,-0.507570,0.016358,-0.019267,0.240645,0.506119,-0.074261,-0.145514,-0.354335,-0.073546,-0.151842,...,0.086176,-0.099357,0.363076,0.155042,-0.015664,0.306506,0.131878,-0.157326,-0.142426,0.246968
AI CROSS株式会社,-0.074489,0.181926,-0.119393,0.048338,0.196564,-0.089658,0.048384,-0.431863,0.137545,-0.524602,...,-0.434743,0.466063,0.287203,-0.277783,0.108197,-0.341298,0.489091,0.373679,-0.303895,-0.228825
AI inside 株式会社,-0.203368,0.022376,0.048530,0.144561,0.310636,0.059621,0.205358,-0.173745,0.098030,-0.529878,...,-0.343483,0.553954,0.150800,-0.055091,0.204542,-0.351584,0.392110,0.121578,-0.183349,-0.202841


In [29]:
#次回のために保存
np.save("../models/true_neo_company_embeddings.npy", final_embeddings)

# 1. 企業名付きデータフレーム（3637, 768）を pickle で保存する
df_final_embeddings.to_pickle('../models/true_neo_company_data_with_sbert.pkl')